In [ ]:
# JupyterLite 环境准备：安装本题依赖，并按需下载本题素材
from pathlib import Path
import asyncio
import hashlib
import piplite
from pyodide.http import pyfetch

PACKAGES = ["pandas", "scikit-learn", "imbalanced-learn"]
ASSETS = [{"name": "finance数据集.csv", "url": "/labs/2.2.1/finance%E6%95%B0%E6%8D%AE%E9%9B%86.csv", "size": 7777795, "sha256": "12aad75e1e14015d97b7b182f29ad7dde37190b56ff87a5f23b21b2912831cd3"}]

async def _retry(label, operation, attempts=3):
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            return await operation()
        except Exception as error:
            last_error = error
            if attempt < attempts:
                await asyncio.sleep(0.8 * attempt)
    raise RuntimeError(f"{label}失败（已重试 {attempts} 次）：{last_error}") from last_error

if PACKAGES:
    await _retry("依赖安装", lambda: piplite.install(PACKAGES))

for asset in ASSETS:
    target = Path(asset["name"])
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and hashlib.sha256(target.read_bytes()).hexdigest() == asset["sha256"]:
        continue
    async def _download():
        separator = "&" if "?" in asset["url"] else "?"
        response = await pyfetch(f'{asset["url"]}{separator}sha256={asset["sha256"][:16]}')
        if response.status < 200 or response.status >= 300:
            raise RuntimeError(f'HTTP {response.status}：{asset["url"]}')
        return await response.bytes()
    payload = await _retry(f'素材下载 {asset["name"]}', _download)
    digest = hashlib.sha256(payload).hexdigest()
    if digest != asset["sha256"]:
        raise RuntimeError(f"素材校验失败：{asset['name']}")
    target.write_bytes(payload)

print("✓ 2.2.1 环境已就绪", f"依赖 {len(PACKAGES)} 项，素材 {len(ASSETS)} 个")


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import pickle
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE

# 加载数据
data = __________

# 显示前五行的数据
print(__________)

# 选择自变量和因变量
X = data.drop(['SeriousDlqin2yrs', 'Unnamed: 0'], axis=1)
y = data['SeriousDlqin2yrs']

# 分割训练集和测试集（测试集20%）
X_train, X_test, y_train, y_test = __________(__________, random_state=42)

# 训练Logistic回归模型（最大迭代次数为1000次）
model = __________
#训练 Logistic 回归模型
__________

# 保存模型
with open('2.2.1_model.pkl', 'wb') as file:
    pickle.__________

# 预测并保存结果
y_pred = __________
pd.DataFrame(y_pred, columns=['预测结果']).to_csv('2.2.1_results.txt', index=False)

# 生成测试报告
report = classification_report(y_test, y_pred, zero_division=1)
with open('2.2.1_report.txt', 'w') as file:
    file.write(report)

# 分析测试结果
accuracy = __________
print(f"模型准确率: {accuracy:.2f}")

# 处理数据不平衡
smote = SMOTE(random_state=42)
X_resampled, y_resampled = __________

# 重新训练模型
__________
# 重新预测
y_pred_resampled = __________

# 保存新结果
pd.DataFrame(y_pred_resampled, columns=['预测结果']).to_csv('2.2.1_results_xg.txt', index=False)

# 生成新的测试报告
report_resampled = classification_report(y_test, y_pred_resampled, zero_division=1)
with open('2.2.1_report_xg.txt', 'w') as file:
    file.write(report_resampled)

# 分析新的测试结果
accuracy_resampled = __________
print(f"重新采样后的模型准确率: {accuracy_resampled:.2f}")
